In [ ]:
from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    PROJECT_ROOT = Path("/content/emotion-dynamics-nlp")

    if not PROJECT_ROOT.exists():
        !git clone https://github.com/duckydodo/emotion-dynamics-nlp.git /content/emotion-dynamics-nlp

else:
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Running in Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)

In [3]:
from pathlib import Path
import sys

if Path("/content").exists():
    PROJECT_ROOT = Path("/content/emotion-dynamics-nlp")
else:
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

Project root: /home/chitta/Projects/emotion-dynamics-nlp
Python: /home/chitta/.conda/envs/emotion-nlp/bin/python


In [4]:
import json
import random

import numpy as np
import pandas as pd
import torch

from src.config import RAW_DATA_DIR, OUTPUT_DIR
from src.data.loader import load_meld
from src.models.transformer import (
    TransformerEmotionClassifier,
    LABELS,
)
from src.models.trainer import TransformerTrainer

print("Imports OK")

Imports OK


In [5]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2,
        ),
        "GB",
    )
else:
    print("Using CPU")

PyTorch: 2.13.0+cu130
CUDA available: False
Using CPU


In [6]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

Seed: 42


In [7]:
EXPERIMENT_NAME = "transformer_baseline"

MODEL_NAME = "distilbert-base-uncased"

LEARNING_RATE = 2e-5
BATCH_SIZE = 4
EPOCHS = 3
MAX_LENGTH = 128

print("Experiment:", EXPERIMENT_NAME)
print("Model:", MODEL_NAME)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Max length:", MAX_LENGTH)

Experiment: transformer_baseline
Model: distilbert-base-uncased
Learning rate: 2e-05
Batch size: 4
Epochs: 3
Max length: 128


In [8]:
dataset = load_meld(
    RAW_DATA_DIR / "meld"
)

train_df = dataset.train
dev_df = dataset.dev
test_df = dataset.test

print("Train:", len(train_df))
print("Dev:", len(dev_df))
print("Test:", len(test_df))

Train: 9989
Dev: 1109
Test: 2610


In [9]:
print("Labels:")
print(sorted(train_df["emotion"].unique()))

print("\nExpected labels:")
print(LABELS)

print("\nTrain distribution:")
print(train_df["emotion"].value_counts())

Labels:
['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

Expected labels:
['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

Train distribution:
emotion
neutral     4710
joy         1743
surprise    1205
anger       1109
sadness      683
disgust      271
fear         268
Name: count, dtype: int64


In [10]:
classifier = TransformerEmotionClassifier(
    model_name=MODEL_NAME,
)

print("Model:", MODEL_NAME)
print("Device:", classifier.device)
print("Labels:", LABELS)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: distilbert-base-uncased
Device: cpu
Labels: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']


In [11]:
TINY_TRAIN_SIZE = 32
TINY_DEV_SIZE = 16

tiny_train = train_df.head(TINY_TRAIN_SIZE)
tiny_dev = dev_df.head(TINY_DEV_SIZE)

tiny_trainer = TransformerTrainer(
    classifier=classifier,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=1,
    max_length=64,
    seed=SEED,
)

tiny_history = tiny_trainer.fit(
    train_texts=tiny_train["text"],
    train_labels=tiny_train["emotion"],
    val_texts=tiny_dev["text"],
    val_labels=tiny_dev["emotion"],
)

print(tiny_history)

Epoch 1/1 | Loss: 1.9007 | Val Accuracy: 0.4375 | Val Macro F1: 0.1217
[{'epoch': 1, 'train_loss': 1.900673732161522, 'val_accuracy': 0.4375, 'val_macro_f1': 0.12173913043478261, 'val_weighted_f1': 0.266304347826087}]


In [15]:
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    CHECKPOINT_DIR = Path(
        "/content/drive/MyDrive/"
        "emotion-dynamics-nlp/"
        "checkpoints/"
        "transformer_baseline"
    )
else:
    CHECKPOINT_DIR = (
        OUTPUT_DIR
        / "models"
        / "transformer_baseline"
    )

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Running in Colab:", IN_COLAB)
print("Checkpoint directory:", CHECKPOINT_DIR)

Running in Colab: False
Checkpoint directory: /home/chitta/Projects/emotion-dynamics-nlp/outputs/models/transformer_baseline


In [13]:
FULL_TRAINING = True

TRAIN_TEXTS = train_df["text"]
TRAIN_LABELS = train_df["emotion"]

VAL_TEXTS = dev_df["text"]
VAL_LABELS = dev_df["emotion"]

print("Training examples:", len(TRAIN_TEXTS))
print("Validation examples:", len(VAL_TEXTS))
print("Full training:", FULL_TRAINING)

Training examples: 9989
Validation examples: 1109
Full training: True


In [14]:
trainer = TransformerTrainer(
    classifier=TransformerEmotionClassifier(
        model_name=MODEL_NAME,
    ),
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    max_length=MAX_LENGTH,
    seed=SEED,
)

print("Full trainer ready")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Full trainer ready


In [ ]:
history = trainer.fit(
    train_texts=TRAIN_TEXTS,
    train_labels=TRAIN_LABELS,
    val_texts=VAL_TEXTS,
    val_labels=VAL_LABELS,
    checkpoint_dir=CHECKPOINT_DIR,
    resume=True,
)

print(history)

In [ ]:
# Load the best model selected using validation Macro F1

best_checkpoint = CHECKPOINT_DIR / "best"

best_classifier = TransformerEmotionClassifier(
    model_name=MODEL_NAME,
)

best_classifier.load(best_checkpoint)

test_trainer = TransformerTrainer(
    classifier=best_classifier,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=1,
    max_length=MAX_LENGTH,
    seed=SEED,
)

test_results, test_probabilities = test_trainer.validate(
    test_df["text"],
    test_df["emotion"],
)

print("===== TEST RESULTS =====")
print(f"Accuracy    : {test_results.accuracy:.4f}")
print(f"Macro F1    : {test_results.macro_f1:.4f}")
print(f"Weighted F1 : {test_results.weighted_f1:.4f}")

In [ ]:
results = {
    "experiment": EXPERIMENT_NAME,
    "model": MODEL_NAME,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "test_accuracy": test_results.accuracy,
    "test_macro_f1": test_results.macro_f1,
    "test_weighted_f1": test_results.weighted_f1,
}

results_path = (
    OUTPUT_DIR
    / "experiments"
    / EXPERIMENT_NAME
    / "metrics.json"
)

results_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved results to:", results_path)